#### Strategy Outline

Bias for commercial net long > 80 and <20 for net short
When bias is long and RSI is <30 go long, go short when bias is short and rsi is  over 70 go short

Exit: RSI @50 or 20 day limit

Risk management: 2 ATR stop, 3 ATR target, 1% risk per trade

In [23]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime, timedelta
import json
import warnings
warnings.filterwarnings('ignore')

with open('cot_data.json', 'r') as f:
    data__raw = json.load(f)

df = pd.DataFrame(data__raw)
df["Date"] = pd.to_datetime(df["Date"], unit='ms')  # Convert from milliseconds


In [24]:
df.head()

,Market,Date,Close,200MA,RSI,YF_Symbol,data_type,group,OI,OI_Index,...,Traders_Index,Net Retail Position,Net Commercial Position,Net Traders Position,Noncommercial Positions-Long (All),Noncommercial Positions-Short (All),Commercial Positions-Long (All),Commercial Positions-Short (All),Nonreportable Positions-Long (All),Nonreportable Positions-Short (All)
0,3 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE,2025-09-02,NaN,NaN,NaN,None,weekly_cot,Financials,15046.0,0.000000,...,0.000000,1.0,-6212.0,6211.0,12194.0,5983.0,10.0,6222.0,2.0,1.0
1,3 YEAR ERIS SOFR SWAP - CHICAGO BOARD OF TRADE,2025-09-09,NaN,NaN,NaN,None,weekly_cot,Financials,22875.0,100.000000,...,100.000000,0.0,-8864.0,8864.0,19567.0,10703.0,10.0,8874.0,1.0,1.0
2,AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,2024-01-02,0.67645,NaN,NaN,6A=F,weekly_cot,Currencies,157003.0,8.566781,...,45.860528,12598.0,30295.0,-42893.0,44442.0,87335.0,82030.0,51735.0,27906.0,15308.0
3,AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,2024-01-02,0.67645,NaN,NaN,6A=F,daily_price,Currencies,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE,2024-01-03,0.67330,NaN,NaN,6A=F,daily_price,Currencies,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


#### Prepare data for analysis

In [25]:
def prepare_strategy_data(df, market_name):

    market_data = df[df['Market'] == market_name].copy()

    cot_weekly = market_data[market_data['data_type'] == 'weekly_cot'].copy()
    price_daily = market_data[market_data['data_type'] == 'daily_price'].copy()

    if price_daily.empty:
        print(f"No daily price data for {market_name}, using weekly data")
        return cot_weekly

    strategy_data = price_daily[['Date', 'Close', 'RSI']].copy()

    # SIMPLE FIX: Just merge weekly COT data directly, then forward fill
    cot_cols = ['Net Commercial Position', 'OI', 'Commercial_Index']
    cot_for_merge = cot_weekly[['Date'] + cot_cols].copy()
    
    strategy_data = pd.merge(strategy_data, cot_for_merge, on='Date', how='left')
    
    # Forward fill COT values to fill gaps between weekly reports
    strategy_data[cot_cols] = strategy_data[cot_cols].fillna(method='ffill')
    
    # Remove rows with missing critical data
    strategy_data = strategy_data.dropna(subset=['Close', 'Commercial_Index'])
    
    return strategy_data.sort_values('Date').reset_index(drop=True) 



In [26]:
#Confirm the function is working using  a random market 
prepare_strategy_data(df, "AUSTRALIAN DOLLAR - CHICAGO MERCANTILE EXCHANGE")

,Date,Close,RSI,Net Commercial Position,OI,Commercial_Index
0,2024-01-02,0.67645,NaN,30295.0,157003.0,48.004386
1,2024-01-03,0.67330,NaN,30295.0,157003.0,48.004386
2,2024-01-04,0.67055,NaN,30295.0,157003.0,48.004386
3,2024-01-05,0.67155,NaN,30295.0,157003.0,48.004386
4,2024-01-08,0.67245,NaN,30295.0,157003.0,48.004386
...,...,...,...,...,...,...
423,2025-09-08,0.65945,71.369249,79114.0,185048.0,77.099164
424,2025-09-09,0.65850,70.347780,73221.0,208092.0,73.587098
425,2025-09-10,0.66210,73.098453,73221.0,208092.0,73.587098
426,2025-09-11,0.66625,75.465392,73221.0,208092.0,73.587098


### Technical Indicators

In [ ]:
#ATR
def calculate_atr(data, period=5):


SyntaxError: incomplete input (248093513.py, line 2)